In [ ]:
# Reference: https://www.kaggle.com/code/guriya79/eda-for-heart-failure 

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# 그래프 기본 테마 설정
sns.set_theme(palette="icefire", style="darkgrid", font_scale=1)
sns.color_palette("icefire", as_cmap=True)

# 그래프를 그리기 위한 기본 설정
plt.rcParams['font.family'] = 'Galmuri9'
# plt.rcParams['font.family'] = 'AppleGothic'
plt.rcParams['figure.figsize'] = 12, 9
plt.rcParams['font.size'] = 16
plt.rcParams['axes.unicode_minus'] = False

In [ ]:
heart_csv = pd.read_csv("data/heart_failure_clinical_records_dataset.csv")
heart_csv

# Dataset info

In [ ]:
heart_csv.info() # 세상에마상에 결측값이 하나도 없어 

In [ ]:
heart_csv.describe()

In [ ]:
heart_csv.columns 
# 전체 칼럼: 'age', 'anaemia', 'creatinine_phosphokinase', 'diabetes', 'ejection_fraction', 'high_blood_pressure', 'platelets', 'serum_creatinine', 'serum_sodium', 'sex', 'smoking', 'time', 'DEATH_EVENT'

In [ ]:
heart_csv.isna().sum() # (기립박수)

In [ ]:
heart_csv.shape # 299, 13

In [ ]:
heart_csv.columns

1. age: 나이
2. anaemia: 빈혈
3. creatinine_phosphokinase: Creatine kinase(크레아티닌 인산화효소)
4. diabetes: 당뇨병
5. ejection_fraction: 박출계수
6. high_blood_pressure: 고혈압
7. platelets: 혈소판
8. serum_creatinine: 혈중 크레아티닌
9. serum_sodium: 혈중 나트륨
10. sex: 성별
11. smoking: 흡연
12. time: 시간(뭐에 대한?)
13. DEATH_EVENT: 사망여부(1이면 멀리 가신 케이스)

In [ ]:
heart_csv.head()

# 전처리
## 나이

In [ ]:
# 나이 범주화
# 나이는 40세~95세까지 있음
main_age_group = [40, 50, 60, 70, 80, 90, 100]
labels = ['40s', '50s', '60s', '70s', '80s', '90s']
heart_csv['age_group'] = pd.cut(heart_csv['age'], bins=main_age_group, labels=labels)

# 의학 데이터 범주화

In [ ]:
# 크레아틴 인산화효소
CK_bins = [0, 200, 250, float('inf')]
CK_labels = ['정상', '주의', '위험']

# 박출계수
EF_bins = [0, 40, 55, float('inf')]
EF_labels = ['위험','주의','정상']

# 혈중 크레아티닌
BC_bins = [0, 1.1, 1.4, float('inf')]
BC_labels = ['정상', '주의', '위험']

# 혈중 나트륨
sodium_bins = [0, 135, 145, 200]
sodium_labels = ['저나트륨(위험)', '정상', '고나트륨']

# 혈소판
plat_bins = [0, 150000, 450000, float('inf')]
plat_labels = ['낮음(위험)', '정상', '높음']

heart_csv['CK_group'] = pd.cut(heart_csv['creatinine_phosphokinase'], bins=CK_bins, labels=CK_labels)
heart_csv['EF_group'] = pd.cut(heart_csv['ejection_fraction'], bins=EF_bins, labels=EF_labels)
heart_csv['BC_group'] = pd.cut(heart_csv['serum_creatinine'], bins=BC_bins, labels=BC_labels)
heart_csv['Na_group'] = pd.cut(heart_csv['serum_sodium'], bins=sodium_bins, labels=sodium_labels)
heart_csv['plate_group'] = pd.cut(heart_csv['platelets'], bins=plat_bins, labels=plat_labels)

In [ ]:
heart_csv.head()

# 이제 가봅시다. 

## 단일 요인들에 따른 사망자 비율

In [ ]:
# 크레아티닌 인산화효소
CK_heart = heart_csv.groupby(['CK_group', 'DEATH_EVENT'], observed=False).size().unstack()

# 생존저 대비 사망자 비율
CK_heart['LD_rate'] = round(CK_heart[1]/CK_heart[0], 2)

CK_heart

In [ ]:
# 박출률
# 40 미만이면 위험
EF_heart = heart_csv.groupby(['EF_group', 'DEATH_EVENT'], observed=False).size().unstack()

# 생존저 대비 사망자 비율
EF_heart['LD_rate'] = round(EF_heart[1]/EF_heart[0], 2)

EF_heart

In [ ]:
# 혈청 크레아티닌
D_heart = heart_csv.groupby(['BC_group', 'DEATH_EVENT'], observed=False).size().unstack()

# 생존저 대비 사망자 비율
D_heart['LD_rate'] = round(D_heart[1]/D_heart[0], 2)

D_heart

In [ ]:
# 나트륨
Na_heart = heart_csv.groupby(['Na_group', 'DEATH_EVENT'], observed=False).size().unstack()

# 생존저 대비 사망자 비율
Na_heart['LD_rate'] = round(Na_heart[1]/Na_heart[0], 2)

Na_heart

- 여러분 너무 짜게 드시는것도 좋지 않지만 너무 심심하게 드시는 것도 좋지 않습니다. 적당한 염분은 필요해요. 

In [ ]:
# 혈소판(딱지 딱지)
pl_heart = heart_csv.groupby(['plate_group', 'DEATH_EVENT'], observed=False).size().unstack()

# 생존저 대비 사망자 비율
pl_heart['LD_rate'] = round(pl_heart[1]/pl_heart[0], 2)

pl_heart

- 이 와중에 혈중 크레아티닌은 비율이 1을 넘어갔음. 

## 복합 요소들

### 혈중 크레아티닌+혈중 나트륨

In [ ]:
# 나트륨+크레아티닌
complex_heart = heart_csv.groupby(['BC_group', 'Na_group', 'DEATH_EVENT'], observed=False).size().unstack()

# 생존저 대비 사망자 비율
complex_heart['LD_rate'] = round(complex_heart[1]/complex_heart[0], 2)

complex_heart

In [ ]:
# 혈중 크레아티닌&혈중 나트륨 히트맵
pivot_df = heart_csv.pivot_table(index='BC_group', 
                                columns='Na_group', 
                                values='DEATH_EVENT', 
                                aggfunc='mean')

plt.figure(figsize=(10, 7))
sns.heatmap(pivot_df, annot=True, fmt='.2f', cmap='YlOrRd', cbar_kws={'label': '사망률'})
plt.title('신장 기능(Creatinine)과 나트륨 수치에 따른 사망률 지도')
plt.xlabel('나트륨 수치 단계')
plt.ylabel('크레아티닌 수치 단계')
plt.show()

- 내 위에도 썼지만... 여러분... 나트륨은 악의 축이 아닙니다... 저나트륨도 죽으니까 적당히 전해질 유지만 하십쇼.. 
- 이거 핑계 대면서 짜게 먹다간 건강검진 결과 나오자마자 의사쌤이 아주 진지한 표정으로 쓰읍 환자분 쓰읍 하시는 걸 보게 될 겁니다. 

In [ ]:
# 평균 생존 시간
complex_heart_time = heart_csv.groupby(['BC_group', 'Na_group', 'DEATH_EVENT'], observed=False)['time'].aggregate('mean')
complex_heart_time.unstack()

# 히트매앱
df_plot = complex_heart_time.reset_index()
df_death = df_plot[df_plot['DEATH_EVENT'] == 1]

plt.figure(figsize=(12, 6))
sns.barplot(data=df_death, x='BC_group', y='time', hue='Na_group', palette='OrRd')

plt.title('신장 기능 및 나트륨 수치별 평균 사망 시간 (DEATH_EVENT=1)')
plt.ylabel('평균 생존 시간 (days)')
plt.xlabel('크레아티닌 수치 단계')
plt.legend(title='나트륨 단계')
plt.show()

- 여러분... 젊을 때 적당히 드십쇼... 적어도 제 명에 살다 가자고요... 

In [ ]:
# 고혈압 필터링
bp_filtered = heart_csv.query('high_blood_pressure == 1')

# 나트륨+크레아티닌
complex_heart = bp_filtered.groupby(['BC_group', 'Na_group', 'DEATH_EVENT'], observed=False).size().unstack()

# 생존저 대비 사망자 비율
complex_heart['LD_rate'] = round(complex_heart[1]/complex_heart[0], 2)

complex_heart

In [ ]:
# 혈중 크레아티닌&혈중 나트륨 히트맵 인데 고혈압이신 분들만 
pivot_df = bp_filtered.pivot_table(index='BC_group', 
                                columns='Na_group', 
                                values='DEATH_EVENT', 
                                aggfunc='mean')

plt.figure(figsize=(10, 7))
sns.heatmap(pivot_df, annot=True, fmt='.2f', cmap='YlOrRd', cbar_kws={'label': '사망률'})
plt.title('신장 기능(Creatinine)과 나트륨 수치에 따른 사망률 지도')
plt.xlabel('나트륨 수치 단계')
plt.ylabel('크레아티닌 수치 단계')
plt.show()

In [ ]:
# 평균 생존 시간
# 평균 생존 시간
complex_heart_time = bp_filtered.groupby(['BC_group', 'Na_group', 'DEATH_EVENT'], observed=False)['time'].aggregate('mean')
complex_heart_time.unstack()

# 히트매앱
df_plot = complex_heart_time.reset_index()
df_death = df_plot[df_plot['DEATH_EVENT'] == 1]

plt.figure(figsize=(12, 6))
sns.barplot(data=df_death, x='BC_group', y='time', hue='Na_group', palette='OrRd')

plt.title('신장 기능 및 나트륨 수치별 평균 사망 시간 (DEATH_EVENT=1)')
plt.ylabel('평균 생존 시간 (days)')
plt.xlabel('크레아티닌 수치 단계')
plt.legend(title='나트륨 단계')
plt.show()

### 혈중 크레아티닌+박출률

In [ ]:
# 박출+크레아티닌
complex_heart = heart_csv.groupby(['BC_group', 'EF_group', 'DEATH_EVENT'], observed=False).size().unstack()

# 생존저 대비 사망자 비율
complex_heart['LD_rate'] = round(complex_heart[1]/complex_heart[0], 2)

complex_heart

In [ ]:
# 혈중 크레아티닌&박출량 히트맵
pivot_df = heart_csv.pivot_table(index='BC_group', 
                                columns='EF_group', 
                                values='DEATH_EVENT', 
                                aggfunc='mean')

plt.figure(figsize=(10, 7))
sns.heatmap(pivot_df, annot=True, fmt='.2f', cmap='YlOrRd', cbar_kws={'label': '사망률'})
plt.title('신장 기능(Creatinine)과 심박출량에 따른 사망률 지도')
plt.xlabel('심박출량')
plt.ylabel('크레아티닌 수치 단계')
plt.show()

In [ ]:
# 평균 생존 시간
complex_heart_time = heart_csv.groupby(['BC_group', 'EF_group', 'DEATH_EVENT'], observed=False)['time'].aggregate('mean')
complex_heart_time.unstack()

# 히트매앱
df_plot = complex_heart_time.reset_index()
df_death = df_plot[df_plot['DEATH_EVENT'] == 1]

plt.figure(figsize=(12, 6))
sns.barplot(data=df_death, x='BC_group', y='time', hue='EF_group', palette='OrRd')

plt.title('신장 기능 및 나트륨 수치별 평균 사망 시간 (DEATH_EVENT=1)')
plt.ylabel('평균 생존 시간 (days)')
plt.xlabel('크레아티닌 수치 단계')
plt.legend(title='나트륨 단계')
plt.show()

In [ ]:
# 고혈압이신 분
# 고혈압 필터링
bp_filtered = heart_csv.query('high_blood_pressure == 1')

# 나트륨+크레아티닌
complex_heart = bp_filtered.groupby(['BC_group', 'EF_group', 'DEATH_EVENT'], observed=False).size().unstack()

# 생존저 대비 사망자 비율
complex_heart['LD_rate'] = round(complex_heart[1]/complex_heart[0], 2)

complex_heart

In [ ]:
# 혈중 크레아티닌&박출량 히트맵
pivot_df = bp_filtered.pivot_table(index='BC_group', 
                                columns='EF_group', 
                                values='DEATH_EVENT', 
                                aggfunc='mean')

plt.figure(figsize=(10, 7))
sns.heatmap(pivot_df, annot=True, fmt='.2f', cmap='YlOrRd', cbar_kws={'label': '사망률'})
plt.title('신장 기능(Creatinine)과 심박출량에 따른 사망률 지도')
plt.xlabel('심박출량')
plt.ylabel('크레아티닌 수치 단계')
plt.show()

In [ ]:
# 평균 생존 시간
complex_heart_time = bp_filtered.groupby(['BC_group', 'EF_group', 'DEATH_EVENT'], observed=False)['time'].aggregate('mean')
complex_heart_time.unstack()

# 히트매앱
df_plot = complex_heart_time.reset_index()
df_death = df_plot[df_plot['DEATH_EVENT'] == 1]

plt.figure(figsize=(12, 6))
sns.barplot(data=df_death, x='BC_group', y='time', hue='EF_group', palette='OrRd')

plt.title('신장 기능 및 나트륨 수치별 평균 사망 시간 (DEATH_EVENT=1)')
plt.ylabel('평균 생존 시간 (days)')
plt.xlabel('크레아티닌 수치 단계')
plt.legend(title='나트륨 단계')
plt.show()

- 크레아티닌 이거 되게 위험한 애였구나... 

In [ ]:
# 1. 숫자형 컬럼만 선택해서 상관계수 계산
numeric_heart = heart_csv.select_dtypes(include=['number'])
corr_matrix = numeric_heart.corr()

# 2. NanumSquare 폰트로 히트맵 그리기
import seaborn as sns
import matplotlib.pyplot as plt

plt.figure(figsize=(12, 10))

# 상관관계 히트맵 시각화
sns.heatmap(corr_matrix, annot=True, fmt='.2f', cmap='coolwarm', linewidths=0.5)
plt.title('심부전 데이터 변수 간 상관관계 (Correlation Heatmap)')
plt.show()

In [ ]:
smoke_pivot = heart_csv.pivot_table(index='sex', 
                                    columns='smoking', 
                                    values='DEATH_EVENT', 
                                    aggfunc=['mean', 'count'])
print(smoke_pivot)

# 시각화로 한눈에 보기
sns.catplot(data=heart_csv, x='smoking', y='DEATH_EVENT', hue='sex', kind='bar', palette='Set2')
plt.title('성별 및 흡연 여부별 사망률')
plt.show()